# 2. parquet-to-iceberg

Con este tomo el parquet que quedó en el bucket `taxis` (paso anterior) y lo guardo como tabla Iceberg en `my-bucket`, registrada en el catálogo de Nessie.

In [1]:
import dlt
import pyarrow.parquet as pq
import s3fs

Nos conectamos a Minio con s3fs para buscar el parquet que dejó el paso 1 en `taxis/taxis_raw/yellow_tripdata/`. El usuario y la clave los saco de `dlt.secrets` (misma sección del secrets.toml del paso 1), para no dejarlos escritos en el notebook.

In [2]:
fs = s3fs.S3FileSystem(
    key=dlt.secrets["http_to_bucket.destination.filesystem.credentials.aws_access_key_id"],
    secret=dlt.secrets["http_to_bucket.destination.filesystem.credentials.aws_secret_access_key"],
    client_kwargs={"endpoint_url": dlt.secrets["http_to_bucket.destination.filesystem.credentials.endpoint_url"]},
)

parquet_path = [p for p in fs.ls("taxis/taxis_raw/yellow_tripdata") if p.endswith(".parquet")][0]
print(parquet_path)

taxis/taxis_raw/yellow_tripdata/1788806023.9298635.31a23a5c91.parquet


La parte importante es el `table_format="iceberg"` del resource, ahí es donde le decimos a dlt que en vez de un parquet plano arme una tabla Iceberg de una vez.

In [3]:
@dlt.resource(name="yellow_tripdata", table_format="iceberg", write_disposition="replace")
def yellow_tripdata_iceberg():
    table = pq.read_table("s3://" + parquet_path, filesystem=fs)
    yield table

`my-bucket` es donde está el warehouse de Nessie (se ve en el docker-compose). El `dataset_name` que le pongamos (`taxis`) es el namespace que va a quedar creado en el catálogo. La conexión a Nessie (sección `iceberg_catalog`) también está en el secrets, no la escribo acá.

In [4]:
pipeline = dlt.pipeline(
    pipeline_name="parquet_to_iceberg",
    destination=dlt.destinations.filesystem(bucket_url="s3://my-bucket"),
    dataset_name="taxis",
)

Corremos el pipeline.

In [5]:
load_info = pipeline.run(yellow_tripdata_iceberg)
print(load_info)

Pipeline parquet_to_iceberg load step finished in 10.02 seconds
1 load package(s) were loaded to destination filesystem and into dataset taxis
The filesystem destination used s3://my-bucket location to store data
Load package 1788806047.037886 is LOADED and contains no failed jobs


Para comprobar que sí quedó registrada en el catálogo de Nessie (y no como un parquet suelto), la vuelvo a leer con `get_catalog` de dlt, que arma el catálogo REST de pyiceberg leyendo la sección `iceberg_catalog` del secrets.toml.

In [6]:
from dlt.common.libs.pyiceberg import get_catalog

catalog = get_catalog()
print("namespaces en nessie:", catalog.list_namespaces())

tabla_iceberg = catalog.load_table("taxis.yellow_tripdata")
print("filas en la tabla iceberg:", tabla_iceberg.scan().to_arrow().num_rows)

namespaces en nessie: [('taxis',)]
filas en la tabla iceberg: 3475226
